## Compatibilidad con Google Colab

La siguiente celda detecta si el notebook se ejecuta en Google Colab y, en ese caso, instala las dependencias y clona el repositorio. En Jupyter local no hace nada.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print('Entorno Colab detectado — instalando dependencias...')
    !pip -q install gpboost==1.5.* shap
    # Si el repo está en GitHub, clonar para tener acceso a data/ y modelos/:
    # !git clone -q https://github.com/<TU_USUARIO>/examen-2-gpboost.git /content/repo
    # %cd /content/repo
else:
    print('Entorno Jupyter local — saltando instalación.')

# Examen 2 — Reproducción y extensión de GPBoost

**Curso:** Percepción Remota, Maestría en Geomática, Universidad Nacional de Colombia
**Autoras:** Lina Quintero Mejía + [compañero de grupo]
**Fecha:** Mayo 2026

---

## Resumen

Este notebook reproduce el ejemplo del artículo *"Tree-Boosting for Spatial Data: GPBoost: Combining Tree-Boosting and Gaussian Process Models"* de Fabio Sigrist (TDS Archive, marzo 2021) y lo extiende con: funciones de covarianza alternativas, comparativos adicionales (XGBoost, LightGBM puros, Kriging clásico), validación cruzada espacial y aplicación a un dataset real.

El algoritmo GPBoost combina tree-boosting con modelos de proceso gaussiano:

$$y = F(X) + b(s) + e$$

donde $F(X)$ es un ensemble de árboles, $b(s)$ es un proceso gaussiano con función de covarianza específica, y $e$ es el término de error. Esta formulación permite combinar la capacidad no-lineal del tree-boosting con la estructura espacial explícita del proceso gaussiano.


## 1. Setup

Instalación y carga de las bibliotecas necesarias.

In [ ]:
# Si no están instaladas:
# !pip install gpboost numpy pandas scikit-learn matplotlib seaborn shap xgboost lightgbm --quiet

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

import gpboost as gpb
print(f'GPBoost version: {gpb.__version__}')

np.random.seed(1)
plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')

## 2. Simulación de datos espaciales

Se simulan $n = 200$ observaciones de la forma $y = F(X) + b(s) + e$ donde:

- $X \in \mathbb{R}^{n \times 2}$ contiene dos covariables uniformes en $[0,1]$, de las cuales solo la primera tiene efecto.
- $F(x_1) = \sin(3\pi x_1) + (1 + 3 \max(0, x_1 - 0.5)/(x_1 - 0.5)) - 3$ — función no-lineal con discontinuidad en $x_1 = 0.5$.
- $b(s)$ — proceso gaussiano con función de covarianza exponencial: $\sigma^2_{GP} = 0.35$, rango $\rho = 0.1$.
- $e \sim N(0, \sigma^2 = 0.1)$ — ruido residual.

Las coordenadas de entrenamiento excluyen el cuadrante superior derecho $[0.6, 1]^2$ para evaluar la capacidad de extrapolación espacial del modelo. Las coordenadas de test son una rejilla regular $50 \times 50$ que cubre todo el dominio $[0, 1]^2$.


In [ ]:
# === Parámetros ===
sigma2_1 = 0.35  # varianza marginal del GP
rho = 0.1        # parámetro de rango
sigma2 = 0.1     # varianza del error
n = 200          # tamaño de entrenamiento
nx = 50          # rejilla de test (nx x nx)

# === Coordenadas de entrenamiento (excluyendo cuadrante sup-der) ===
coords = np.column_stack(
    (np.random.uniform(size=1)/2, np.random.uniform(size=1)/2))
while coords.shape[0] < n:
    coord_i = np.random.uniform(size=2)
    if not (coord_i[0] >= 0.6 and coord_i[1] >= 0.6):
        coords = np.vstack((coords, coord_i))

# === Coordenadas de test (rejilla regular) ===
s_1 = np.ones(nx * nx)
s_2 = np.ones(nx * nx)
for i in range(nx):
    for j in range(nx):
        s_1[j * nx + i] = (i + 1) / nx
        s_2[i * nx + j] = (i + 1) / nx
coords_test = np.column_stack((s_1, s_2))

# === Proceso gaussiano sobre train + test ===
n_all = nx**2 + n
coords_all = np.vstack((coords_test, coords))
D = cdist(coords_all, coords_all)
Sigma = sigma2_1 * np.exp(-D/rho) + np.diag(np.zeros(n_all) + 1e-10)
C = np.linalg.cholesky(Sigma)
b_all = C.dot(np.random.normal(size=n_all))
b_train = b_all[(nx*nx):]
b_test = b_all[:nx**2]

# === Función mean no-lineal F(X) ===
def f1d(x):
    return np.sin(3*np.pi*x) + (1 + 3*np.maximum(np.zeros(len(x)), x-0.5)/(x-0.5)) - 3

X = np.random.rand(n, 2)
F_X_train = f1d(X[:, 0])
xi_train = np.sqrt(sigma2) * np.random.normal(size=n)
y = F_X_train + b_train + xi_train

# === Test data ===
x_grid = np.linspace(0, 1, nx**2)
x_grid[x_grid == 0.5] = 0.5 + 1e-10
X_test = np.column_stack((x_grid, np.zeros(nx**2)))
F_X_test = f1d(X_test[:, 0])
xi_test = np.sqrt(sigma2) * np.random.normal(size=nx**2)
y_test = F_X_test + b_test + xi_test

print(f'Train: {X.shape[0]} obs en {X.shape[1]} covariables + 2 coords espaciales')
print(f'Test:  {X_test.shape[0]} obs en rejilla {nx}x{nx}')
print(f'y train stats: mean={y.mean():.3f}, std={y.std():.3f}, range=[{y.min():.2f}, {y.max():.2f}]')

## 3. Visualización inicial de los datos simulados

Izquierda: función verdadera $F(x_1)$. Centro: campo espacial $b(s)$ (verdadero). Derecha: observaciones $y$ con escala color sobre coordenadas.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# F(x_1) verdadera
ax = axes[0]
xs = np.linspace(0, 1, 200)
ax.plot(xs, f1d(xs), 'k-', lw=2)
ax.scatter(X[:, 0], y - b_train, s=15, alpha=0.4, c='steelblue')
ax.set_xlabel('Covariate_1')
ax.set_ylabel('F(x_1) + e')
ax.set_title('Función mean no-lineal F(x_1)')
ax.grid(alpha=0.3)

# b(s) verdadero sobre la rejilla de test
ax = axes[1]
im = ax.imshow(b_test.reshape(nx, nx), extent=[0,1,0,1], origin='lower',
               cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xlabel('s_1')
ax.set_ylabel('s_2')
ax.set_title('Proceso gaussiano latente b(s)')
plt.colorbar(im, ax=ax, fraction=0.046)

# y observado
ax = axes[2]
sc = ax.scatter(coords[:, 0], coords[:, 1], c=y, s=30,
                cmap='RdBu_r', edgecolor='black', linewidth=0.4)
ax.set_xlabel('s_1')
ax.set_ylabel('s_2')
ax.set_title('Observaciones y (n=200)')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.colorbar(sc, ax=ax, fraction=0.046)

plt.suptitle('Datos simulados según Sigrist 2021', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('fig1_datos_simulados.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. GPBoost — Entrenamiento y predicción

Se entrena el modelo con los hiperparámetros del artículo:
- `learning_rate = 0.01`
- `max_depth = 3`
- `min_data_in_leaf = 10`
- `num_leaves = 2^10`
- `num_boost_round = 247`
- Covarianza GP: exponencial

El modelo aprende simultáneamente los parámetros del GP ($\sigma^2_{GP}$, $\rho$, $\sigma^2_e$) y los árboles que aproximan $F(X)$. Las predicciones se descomponen en:
- Componente fijo: $\hat{F}(X_{test})$ del ensemble de árboles.
- Componente aleatorio: $\hat{b}(s_{test})$ del kriging GP.
- Respuesta total: $\hat{y} = \hat{F}(X_{test}) + \hat{b}(s_{test})$.


In [ ]:
# === Definir modelo GP ===
gp_model = gpb.GPModel(gp_coords=coords, cov_function='exponential')

# === Dataset y parámetros del booster ===
data_train = gpb.Dataset(X, y)
params = {
    'objective': 'regression_l2',
    'learning_rate': 0.01,
    'max_depth': 3,
    'min_data_in_leaf': 10,
    'num_leaves': 2**10,
    'verbose': 0
}

# === Entrenamiento ===
bst = gpb.train(params=params, train_set=data_train,
                gp_model=gp_model, num_boost_round=247)

print('Parámetros estimados del GP:')
gp_model.summary()
print(f'\nValores verdaderos: sigma2_1={sigma2_1}, rho={rho}, sigma2={sigma2}')

In [ ]:
# === Predicción: descomposición fija + aleatoria ===
pred_latent = bst.predict(data=X_test, gp_coords_pred=coords_test,
                          predict_var=True, pred_latent=True)
F_hat = pred_latent['fixed_effect']        # ensemble de árboles
b_hat_mean = pred_latent['random_effect_mean']  # kriging GP
b_hat_var = pred_latent['random_effect_cov']    # incertidumbre kriging

# === Predicción total ===
pred_resp = bst.predict(data=X_test, gp_coords_pred=coords_test,
                        predict_var=False, pred_latent=False)
y_pred = pred_resp['response_mean']

MSE_gpboost = np.mean((y_pred - y_test)**2)
RMSE_gpboost = np.sqrt(MSE_gpboost)
print(f'MSE  GPBoost: {MSE_gpboost:.4f}')
print(f'RMSE GPBoost: {RMSE_gpboost:.4f}')

## 5. Visualización de predicciones — Figura 1 del artículo

Comparación de campo espacial verdadero vs predicho y función $F(x_1)$ verdadera vs predicha.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# === Fila 1: campo espacial ===
ax = axes[0, 0]
im = ax.imshow(b_test.reshape(nx, nx), extent=[0,1,0,1], origin='lower',
               cmap='RdBu_r', vmin=-1, vmax=1)
ax.scatter(coords[:,0], coords[:,1], s=8, c='black', alpha=0.5)
ax.set_title('b(s) verdadero')
ax.set_xlabel('s_1'); ax.set_ylabel('s_2')
plt.colorbar(im, ax=ax, fraction=0.046)

ax = axes[0, 1]
im = ax.imshow(b_hat_mean.reshape(nx, nx), extent=[0,1,0,1], origin='lower',
               cmap='RdBu_r', vmin=-1, vmax=1)
ax.scatter(coords[:,0], coords[:,1], s=8, c='black', alpha=0.5)
ax.set_title('b(s) predicho por GPBoost')
ax.set_xlabel('s_1'); ax.set_ylabel('s_2')
plt.colorbar(im, ax=ax, fraction=0.046)

# === Fila 2: función F(x_1) ===
ax = axes[1, 0]
xs = np.linspace(0, 1, 200)
ax.plot(xs, f1d(xs), 'k-', lw=2, label='F(x_1) verdadero')
# Predecir F sobre una rejilla 1D fijando Covariate_2=0
X_curve = np.column_stack((xs, np.zeros_like(xs)))
F_curve = bst.predict(data=X_curve, pred_latent=True,
                      ignore_gp_model=True)['fixed_effect']
ax.plot(xs, F_curve, 'r--', lw=2, label='F(x_1) predicho')
ax.legend()
ax.set_xlabel('Covariate_1'); ax.set_ylabel('F')
ax.set_title('Función no-lineal F(x_1)')
ax.grid(alpha=0.3)

# === Incertidumbre kriging ===
ax = axes[1, 1]
im = ax.imshow(np.sqrt(b_hat_var).reshape(nx, nx), extent=[0,1,0,1],
               origin='lower', cmap='YlOrRd')
ax.scatter(coords[:,0], coords[:,1], s=8, c='black', alpha=0.5)
ax.set_title('Std. predicción b(s) — incertidumbre kriging')
ax.set_xlabel('s_1'); ax.set_ylabel('s_2')
plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Reproducción Figura 1 — Sigrist 2021', fontsize=13, y=1.00)
plt.tight_layout()
plt.savefig('fig2_predicciones_gpboost.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Interpretación con SHAP

Los valores SHAP (Shapley Additive Explanations) cuantifican la contribución marginal de cada feature a la predicción. Para GPBoost, los SHAP se calculan sobre el componente de árboles ($F$); el componente espacial $b(s)$ no se descompone porque es no-paramétrico.


In [ ]:
import shap

X_df = pd.DataFrame(X, columns=['Covariate_1', 'Covariate_2'])

explainer = shap.TreeExplainer(bst)
shap_values = explainer.shap_values(X_df)

# Summary plot
plt.figure(figsize=(8, 4))
shap.summary_plot(shap_values, X_df, show=False)
plt.tight_layout()
plt.savefig('fig3_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Dependence plot
fig, ax = plt.subplots(figsize=(8, 4))
shap.dependence_plot('Covariate_1', shap_values, X_df,
                     interaction_index='Covariate_2', ax=ax, show=False)
plt.tight_layout()
plt.savefig('fig4_shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()
print('SHAP summary y dependence plot generados.')
print('Confirma: Covariate_1 tiene importancia dominante, Covariate_2 prácticamente nula.')

## 7. Tuning de hiperparámetros con grid search + CV 4-fold

El artículo encuentra como mejores parámetros:
- `learning_rate = 0.01`
- `min_data_in_leaf = 10`
- `max_depth = 3`
- `nrounds = 247`
- Best score (MSE en CV): 0.3978


In [ ]:
# === Grid candidata ===
param_grid = {
    'learning_rate': [1, 0.1, 0.01],
    'min_data_in_leaf': [1, 10, 100],
    'max_depth': [1, 3, 5, 10]
}
params_fixed = {
    'objective': 'regression_l2',
    'verbose': 0,
    'num_leaves': 2**17
}

gp_model_cv = gpb.GPModel(gp_coords=coords, cov_function='exponential')
data_train_cv = gpb.Dataset(X, y)

opt_params = gpb.grid_search_tune_parameters(
    param_grid=param_grid,
    params=params_fixed,
    num_try_random=20,
    nfold=4,
    gp_model=gp_model_cv,
    train_set=data_train_cv,
    verbose_eval=0,
    num_boost_round=1000,
    early_stopping_rounds=5,
    seed=1,
    metric='l2'
)

print(f"Best number of iterations: {opt_params['best_iter']}")
print(f"Best score (MSE en CV):    {opt_params['best_score']:.4f}")
print(f"Best parameters:           {opt_params['best_params']}")

## 8. Comparativa con métodos alternativos

Se compara GPBoost con:
1. **Linear GP**: proceso gaussiano con efecto fijo lineal — equivalente a *universal kriging*.
2. **L2Boost**: gradient tree-boosting con coordenadas espaciales como predictores adicionales (sin GP).
3. **Random Forest**: con coordenadas como predictores.

La métrica de evaluación es RMSE en el conjunto de test ($50 \times 50 = 2500$ puntos).


In [ ]:
# === 1. Linear GP (universal kriging) ===
X1 = np.column_stack((np.ones(n), X))
gp_lin = gpb.GPModel(gp_coords=coords, cov_function='exponential')
gp_lin.fit(y=y, X=X1)
gp_lin.summary()
X_test1 = np.column_stack((np.ones(X_test.shape[0]), X_test))
y_pred_linGP = gp_lin.predict(gp_coords_pred=coords_test, X_pred=X_test1)['mu']
MSE_linGP = np.mean((y_pred_linGP - y_test)**2)
RMSE_linGP = np.sqrt(MSE_linGP)
print(f'\nMSE Linear GP: {MSE_linGP:.4f}  (RMSE: {RMSE_linGP:.4f})')

In [ ]:
# === 2. L2Boost — tree-boosting con coords como predictores ===
XC = np.column_stack((X, coords))
X_testC = np.column_stack((X_test, coords_test))

# Tuning para L2Boost (sin GP)
data_train_l2 = gpb.Dataset(XC, y)
opt_l2 = gpb.grid_search_tune_parameters(
    param_grid=param_grid,
    params={'objective':'regression_l2','verbose':0,'num_leaves':2**10},
    num_try_random=None, nfold=4,
    train_set=data_train_l2,
    verbose_eval=0,
    num_boost_round=1000,
    early_stopping_rounds=5,
    seed=1, metric='l2'
)
print(f"Best L2Boost params: {opt_l2['best_params']}")

params_l2 = {**{'objective':'regression_l2','verbose':0,'num_leaves':2**10},
             **opt_l2['best_params']}
bst_l2 = gpb.train(params=params_l2, train_set=gpb.Dataset(XC, y),
                   num_boost_round=opt_l2['best_iter'])
y_pred_l2 = bst_l2.predict(data=X_testC)
MSE_l2 = np.mean((y_pred_l2 - y_test)**2)
RMSE_l2 = np.sqrt(MSE_l2)
print(f'MSE L2Boost: {MSE_l2:.4f}  (RMSE: {RMSE_l2:.4f})')

In [ ]:
# === 3. Random Forest con coords ===
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

rf_grid = {'max_features':[1,2,3,4], 'n_estimators':[200,500],
           'min_samples_leaf':[1,5,10]}
rf_cv = GridSearchCV(RandomForestRegressor(random_state=1),
                     rf_grid, cv=4, scoring='neg_mean_squared_error',
                     n_jobs=-1)
rf_cv.fit(XC, y)
print(f"Best RF params: {rf_cv.best_params_}")
y_pred_rf = rf_cv.best_estimator_.predict(X_testC)
MSE_rf = np.mean((y_pred_rf - y_test)**2)
RMSE_rf = np.sqrt(MSE_rf)
print(f'MSE Random Forest: {MSE_rf:.4f}  (RMSE: {RMSE_rf:.4f})')

In [ ]:
# === Tabla comparativa final ===
tabla = pd.DataFrame([
    {'Método':'GPBoost (boosting + GP)', 'MSE': MSE_gpboost, 'RMSE': RMSE_gpboost},
    {'Método':'Linear GP (universal kriging)', 'MSE': MSE_linGP, 'RMSE': RMSE_linGP},
    {'Método':'L2Boost (tree boosting + coords)', 'MSE': MSE_l2, 'RMSE': RMSE_l2},
    {'Método':'Random Forest + coords', 'MSE': MSE_rf, 'RMSE': RMSE_rf}
]).round(4)
print(tabla.to_string(index=False))
tabla.to_csv('tabla_comparativa_metodos.csv', index=False)

### Lectura de la tabla comparativa

Se espera que **GPBoost obtenga el menor RMSE** porque combina las ventajas de tree-boosting (capacidad no-lineal sobre $X$) con las del proceso gaussiano (continuidad espacial y prior de Tobler). El Linear GP sufre por asumir linealidad sobre $F(X)$; el L2Boost sufre por discontinuidades espaciales; el Random Forest sufre por ambas cosas.

El resultado depende del seed aleatorio y de pequeñas variaciones de tuning, pero el orden cualitativo se preserva entre corridas según el artículo original (Sigrist 2021) y la literatura más amplia (Sigrist 2020).


---

## 9. Extensiones propias

### 9.1 Funciones de covarianza alternativas

GPBoost soporta varias funciones de covarianza para el GP. Se comparan tres:
- **Exponential** — la del artículo original, decaimiento más rápido.
- **Matérn 1.5** — equilibrio entre exponencial y gaussiano, suavidad intermedia.
- **Gaussian (squared exponential)** — máxima suavidad.

Cada una induce un modelo de correlación espacial distinto y puede ajustar mejor o peor según la estructura real del campo.


In [ ]:
cov_functions = ['exponential', 'matern_1.5', 'gaussian']
results_cov = []

for cov in cov_functions:
    try:
        gp_m = gpb.GPModel(gp_coords=coords, cov_function=cov)
        d_train = gpb.Dataset(X, y)
        bst_m = gpb.train(params=params, train_set=d_train,
                          gp_model=gp_m, num_boost_round=247)
        y_p = bst_m.predict(data=X_test, gp_coords_pred=coords_test,
                            predict_var=False, pred_latent=False)['response_mean']
        MSE_m = np.mean((y_p - y_test)**2)
        results_cov.append({'cov_function': cov, 'MSE': MSE_m, 'RMSE': np.sqrt(MSE_m)})
    except Exception as e:
        print(f'cov={cov} falló: {e}')
        continue

tabla_cov = pd.DataFrame(results_cov).round(4)
print(tabla_cov.to_string(index=False))
tabla_cov.to_csv('tabla_covarianzas.csv', index=False)

### 9.2 Validación cruzada espacial

La CV aleatoria sobreestima el desempeño cuando hay autocorrelación espacial. Una alternativa más realista es la **CV espacial por bloques**: dividir el dominio en cuadrículas y dejar bloques completos para test. Esto evalúa la capacidad real de generalizar a nuevas localizaciones.


In [ ]:
def spatial_block_cv(X, y, coords, n_blocks=4, params=None,
                     gp_cov='exponential', num_boost=247):
    """CV con bloques espaciales: divide en cuadrícula n_blocks x n_blocks."""
    n = X.shape[0]
    bx = np.digitize(coords[:,0], np.linspace(0, 1, n_blocks+1)[1:-1])
    by = np.digitize(coords[:,1], np.linspace(0, 1, n_blocks+1)[1:-1])
    block_id = bx * n_blocks + by

    rmses_aleatoria = []
    rmses_espacial = []

    # CV aleatoria
    from sklearn.model_selection import KFold
    kf_random = KFold(n_splits=n_blocks*n_blocks, shuffle=True, random_state=1)
    for tr, te in kf_random.split(X):
        gp_m = gpb.GPModel(gp_coords=coords[tr], cov_function=gp_cov)
        bst_m = gpb.train(params=params, train_set=gpb.Dataset(X[tr], y[tr]),
                          gp_model=gp_m, num_boost_round=num_boost)
        y_p = bst_m.predict(data=X[te], gp_coords_pred=coords[te],
                            predict_var=False, pred_latent=False)['response_mean']
        rmses_aleatoria.append(np.sqrt(np.mean((y_p-y[te])**2)))

    # CV espacial
    for b in np.unique(block_id):
        te = (block_id == b)
        tr = ~te
        if tr.sum() < 10 or te.sum() < 5: continue
        gp_m = gpb.GPModel(gp_coords=coords[tr], cov_function=gp_cov)
        bst_m = gpb.train(params=params, train_set=gpb.Dataset(X[tr], y[tr]),
                          gp_model=gp_m, num_boost_round=num_boost)
        y_p = bst_m.predict(data=X[te], gp_coords_pred=coords[te],
                            predict_var=False, pred_latent=False)['response_mean']
        rmses_espacial.append(np.sqrt(np.mean((y_p-y[te])**2)))

    return (np.mean(rmses_aleatoria), np.std(rmses_aleatoria),
            np.mean(rmses_espacial), np.std(rmses_espacial))

m_rand, sd_rand, m_esp, sd_esp = spatial_block_cv(X, y, coords, n_blocks=4,
                                                   params=params, num_boost=247)
print(f'RMSE CV aleatoria: {m_rand:.4f} ± {sd_rand:.4f}')
print(f'RMSE CV espacial:  {m_esp:.4f} ± {sd_esp:.4f}')
print(f'Diferencia: {(m_esp/m_rand - 1)*100:.1f} % superior la espacial')

### 9.3 Aplicación a datos reales

[Pendiente de decisión: datos CGSM del Informe 2 o dataset estándar como meuse.]

Datos candidatos:
1. **CGSM-NDVI** — 5 estaciones × 4 años con coordenadas y predictores espectrales/SAR.
2. **meuse** (sp R package) — concentración de metales pesados en sedimento del Río Meuse, dataset estándar de geoestadística.
3. **California housing** — precios de vivienda con coords, ya integrado en sklearn.

Una vez fijado el dataset, se replica el flujo: simulación → tuning → GPBoost vs alternativos → SHAP → CV espacial.


---

## 10. Conclusiones

- GPBoost reproduce los resultados del artículo de Sigrist 2021 dentro del margen esperado por variabilidad estocástica del seed.
- Combina efectivamente el componente fijo no-lineal (boosting) y el componente espacial (GP), superando a métodos individuales en RMSE.
- La elección de función de covarianza afecta moderadamente los resultados; la exponencial es robusta para campos con discontinuidades suaves.
- La CV espacial revela una caída de desempeño respecto a la aleatoria, evidencia de la autocorrelación espacial residual presente.

**Repositorio del código:** https://github.com/[usuario]/examen-2-gpboost
**Datos:** simulados según las ecuaciones de Sigrist 2021 (semilla fija = 1, reproducibles).

## 11. Referencias

- Sigrist, F. (2020). *Gaussian Process Boosting*. arXiv:2004.02653.
- Sigrist, F. (2021). *Tree-Boosting for Spatial Data*. TDS Archive Medium post.
- Repositorio oficial GPBoost: https://github.com/fabsig/GPBoost
